# OSS 4008 — Train, evaluate, and export hospital object detector

**Status: runnable training/evaluation scaffold, NOT trained yet.**

This notebook trains a YOLO object detector only when a real annotated dataset is present at `/content/hospital_yolo/data.yaml` (the output of Notebook 05). No fabricated accuracy, model weights, or hospital detections are provided.

**Methodology:** Bounding boxes of verified hospital buildings, negative imagery patches, site-disjoint train/validation/test, and image acquisition/license documentation. Detecting hospital *function* from visible building shape alone is fundamentally uncertain.

In [ ]:
%pip -q install "ultralytics>=8.3,<9" pyyaml

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import yaml
ROOT = Path('/content/hospital_yolo')
CONFIG = ROOT / 'data.yaml'
if not CONFIG.exists():
    print('BLOCKED: No verified annotated dataset available yet.')
    print('Complete Notebook 05 or place a valid YOLO dataset in:', ROOT)
else:
    dataset = yaml.safe_load(CONFIG.read_text(encoding='utf-8'))
    for split in ['train','val','test']:
        images_dir = ROOT / dataset.get(split, f'images/{split}')
        count = len(list(images_dir.glob('*'))) if images_dir.exists() else 0
        print(split, ':', count, 'image files')
    print('Dataset config found:', CONFIG)

In [ ]:
# Training is intentionally gated on real annotated data.
TRAIN_MODEL = False  # Set True only AFTER annotation validation and site-disjoint split review.
MODEL_PATH = Path('/content/hospital_detector/best.pt')
if TRAIN_MODEL and CONFIG.exists():
    model = YOLO('yolo11n.pt')
    run = model.train(data=str(CONFIG), epochs=30, imgsz=640, batch=8, seed=4008,
                      project='/content/hospital_training', name='baseline', exist_ok=True)
    best_path = Path(run.save_dir) / 'weights' / 'best.pt'
    MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    import shutil
    shutil.copy2(best_path, MODEL_PATH)
    print('Trained model:', MODEL_PATH)
else:
    print('NO TRAINING RUN: annotations are not yet verified or training is disabled.')

In [ ]:
if MODEL_PATH.exists() and CONFIG.exists():
    model = YOLO(str(MODEL_PATH))
    metrics = model.val(data=str(CONFIG), split='test', imgsz=640)
    print('Test mAP50:', float(metrics.box.map50))
    print('Test mAP50-95:', float(metrics.box.map))
else:
    print('NO EVALUATION: trained weights and/or real test dataset missing.')

### Inference/export contract
The web demo reads a GeoJSON **FeatureCollection of Point predictions**, each with WGS84 longitude/latitude coordinates and optional `confidence` between 0 and 1. Converting detector pixel boxes into ground coordinates requires **georeferenced imagery** with a valid raster affine transform and CRS. This notebook intentionally does not assign invented coordinates to image pixels.

In [ ]:
# Georeferenced inference helper (requires rasterio, a GeoTIFF, and trained weights).
def prediction_geojson_from_geotiff(model_path, geotiff_path, confidence=0.35):
    import json
    import numpy as np
    import rasterio
    from rasterio.warp import transform as convert_crs
    from PIL import Image
    model = YOLO(str(model_path))
    features = []
    with rasterio.open(geotiff_path) as src:
        if src.crs is None:
            raise ValueError('GeoTIFF has no CRS: cannot map pixel predictions to Earth coordinates')
        if src.count < 3:
            raise ValueError('Need at least three image bands')
        rgb = src.read([1,2,3]).transpose(1,2,0)
        if rgb.dtype != np.uint8:
            raise ValueError('Input GeoTIFF must contain display-ready uint8 RGB bands; scale reflectance first')
        results = model.predict(source=Image.fromarray(rgb), conf=confidence, verbose=False)
        for result in results:
            for box in result.boxes:
                x1,y1,x2,y2 = [float(v) for v in box.xyxy[0].tolist()]
                px,py = (x1+x2)/2,(y1+y2)/2
                x,y = src.transform*(px,py)
                lon,lat = convert_crs(src.crs, 'EPSG:4326', [x], [y])
                features.append({'type':'Feature','geometry':{'type':'Point','coordinates':[lon[0],lat[0]]},
                                'properties':{'class':'hospital','confidence':float(box.conf[0]),
                                              'source':'trained YOLO model','bbox_pixels':[x1,y1,x2,y2]}})
    return {'type':'FeatureCollection','features':features}

print('Georeferenced export function defined. No fake predictions produced.')

### Validation and reporting gates
Before claiming hospital detection is complete:
- Record image source, ground sample distance (GSD), dates, geographic area, and usage permission.
- Independently verify hospital labels and include other institutional buildings as negatives.
- Prevent nearby tiles of the same campus from appearing in both train and test.
- Report precision, recall, mAP50, mAP50–95, failure cases and qualitative inspection.
- Run `prediction_geojson_from_geotiff(...)` only on actual georeferenced RGB GeoTIFFs after training.
- Load the resulting GeoJSON in `04_canada_web_demo.ipynb` to show genuine model predictions distinctly from OSM records.